# 08 - Documentation Export
**Fingo Income Predictor** | Tim CC26-PSU217

Generate dokumentasi otomatis:
- `notebook.md`
- `data_dictionary.md`
- `README.md`

Notebook ini dipakai sebagai tahap final untuk memastikan dokumentasi repo selalu sinkron dengan pipeline notebook 01 sampai 07.

In [1]:
# GIT PULL — Sinkronisasi terbaru dari remote sebelum mulai
import os, shutil, subprocess

try:
    from google.colab import userdata
except Exception:
    userdata = None

os.chdir("/content")

GITHUB_USERNAME = "ClarisyaA"
REPO_NAME       = "fingo-income-analysis"
BRANCH_NAME     = "feat/income-predictor-final"
LOCAL_DIR       = f"/content/{REPO_NAME}"
FRESH_CLONE     = False  # Set True hanya kalau mau clone ulang dari nol

def get_remote_url():
    try:
        token = userdata.get("GITHUB_TOKEN") if userdata else os.environ.get("GITHUB_TOKEN", "")
        if token:
            return f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git", token
    except Exception:
        pass
    return f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git", None

remote_url, token = get_remote_url()

def mask_cmd(cmd):
    return cmd.replace(token, "***TOKEN***") if token else cmd

def run_cmd(cmd, check=True, cwd="/content"):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    print(f"$ {mask_cmd(cmd)}")
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command gagal: {mask_cmd(cmd)}")
    return r

def remote_branch_exists():
    r = run_cmd(f"git ls-remote --heads {remote_url} {BRANCH_NAME}", check=False)
    return r.stdout.strip() != ""

branch_exists = remote_branch_exists()

if FRESH_CLONE and os.path.exists(LOCAL_DIR):
    os.chdir("/content")
    shutil.rmtree(LOCAL_DIR)

if not os.path.exists(LOCAL_DIR):
    if branch_exists:
        run_cmd(f"git clone -b {BRANCH_NAME} {remote_url} {LOCAL_DIR}")
    else:
        run_cmd(f"git clone {remote_url} {LOCAL_DIR}")
        run_cmd(f"git checkout -b {BRANCH_NAME}", cwd=LOCAL_DIR)
else:
    run_cmd(f"git remote set-url origin {remote_url}", cwd=LOCAL_DIR)
    run_cmd("git fetch origin", cwd=LOCAL_DIR)
    if branch_exists:
        local_b = run_cmd(f"git branch --list {BRANCH_NAME}", check=False, cwd=LOCAL_DIR).stdout.strip()
        if local_b:
            run_cmd(f"git checkout {BRANCH_NAME}", cwd=LOCAL_DIR)
        else:
            run_cmd(f"git checkout -b {BRANCH_NAME} origin/{BRANCH_NAME}", cwd=LOCAL_DIR)
        run_cmd(f"git pull --rebase origin {BRANCH_NAME}", cwd=LOCAL_DIR)
    else:
        current_branch = run_cmd("git branch --show-current", check=False, cwd=LOCAL_DIR).stdout.strip()
        if current_branch != BRANCH_NAME:
            local_b = run_cmd(f"git branch --list {BRANCH_NAME}", check=False, cwd=LOCAL_DIR).stdout.strip()
            if local_b:
                run_cmd(f"git checkout {BRANCH_NAME}", cwd=LOCAL_DIR)
            else:
                run_cmd(f"git checkout -b {BRANCH_NAME}", cwd=LOCAL_DIR)

os.chdir(LOCAL_DIR)
run_cmd(f"git remote set-url origin {remote_url}", cwd=LOCAL_DIR)
print("\nRepo siap digunakan")
print(f"Working directory: {os.getcwd()}")
run_cmd("git branch --show-current", cwd=LOCAL_DIR)
run_cmd("git status --short", check=False, cwd=LOCAL_DIR)


$ git ls-remote --heads https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git feat/income-predictor-final
4e8574491685ca5995b80da29246cb520f2dc096	refs/heads/feat/income-predictor-final
$ git clone -b feat/income-predictor-final https://***TOKEN***@github.com/ClarisyaA/fingo-income-analysis.git /content/fingo-income-analysis
Cloning into '/content/fingo-income-analysis'...
Updating files:  15% (15/95)
Updating files:  16% (16/95)
Updating files:  17% (17/95)
Updating files:  18% (18/95)
Updating files:  20% (19/95)
Updating files:  21% (20/95)
Updating files:  22% (21/95)
Updating files:  23% (22/95)
Updating files:  24% (23/95)
Updating files:  25% (24/95)
Updating files:  26% (25/95)
Updating files:  27% (26/95)
Updating files:  28% (27/95)
Updating files:  29% (28/95)
Updating files:  30% (29/95)
Updating files:  31% (30/95)
Updating files:  32% (31/95)
Updating files:  33% (32/95)
Updating files:  34% (33/95)
Updating files:  35% (34/95)
Updating files:  36% (35/95)
Up

CompletedProcess(args='git status --short', returncode=0, stdout='', stderr='')

In [2]:
# CELL 08.2 — Setup
import os, json, warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
os.chdir("/content/fingo-income-analysis")

def ensure_dir(path):
    d = os.path.dirname(path) if '.' in os.path.basename(path) else path
    if d:
        os.makedirs(d, exist_ok=True)

def safe_to_csv(df, path, **kwargs):
    ensure_dir(path)
    df.to_csv(path, index=kwargs.pop('index', False), **kwargs)

print('Setup selesai')


Setup selesai


In [3]:
# CELL 08.3 — Load model contract untuk referensi
try:
    with open('outputs/model_contract/model_contract.json', encoding='utf-8') as f:
        model_contract = json.load(f)

    n_features = model_contract.get('n_features', 'N/A')
    feature_list = model_contract.get('feature_columns', [])

    print(f'Model contract dimuat: {n_features} features')

except Exception as e:
    print(f'Model contract tidak ditemukan: {e}')
    model_contract = {}
    n_features = 'N/A'
    feature_list = []


Model contract dimuat: 58 features


In [4]:
# CELL 08.4 — Generate data_dictionary.md (termasuk temporal mapping)
data_dict_md = """# Data Dictionary — Fingo Income Predictor
**Tim:** CC26-PSU217 | **Versi:** v13-FINAL | **Tanggal:** Mei 2026

---

## Temporal Mapping untuk Data Survey

Kolom `income_w1`, `income_w2`, `income_w3`, dan `income_w4` berasal dari pertanyaan
pendapatan 1–4 minggu terakhir pada Google Form.

Karena responden mengisi form pada rentang tanggal tertentu (Mei 2026), periode pendapatan
historis bersifat **relatif terhadap `timestamp` masing-masing responden**.

### Definisi
| Kolom | Periode | Rentang |
|-------|---------|---------|
| `income_w1` | Pendapatan 1 minggu terakhir | H-7 sampai H-1 dari timestamp |
| `income_w2` | Pendapatan 2 minggu terakhir | H-14 sampai H-8 dari timestamp |
| `income_w3` | Pendapatan 3 minggu terakhir | H-21 sampai H-15 dari timestamp |
| `income_w4` | Pendapatan 4 minggu terakhir | H-28 sampai H-22 dari timestamp |

### Urutan Kronologis untuk Model Forecasting
```
income_w4 → income_w3 → income_w2 → income_w1
(terlama)                             (terbaru)
```

> Kolom ini **tidak** merepresentasikan minggu 1–4 dalam satu bulan kalender secara langsung.
> Melainkan 4 periode mingguan relatif sebelum responden mengisi form.

### Kolom Hasil Temporal Mapping
Untuk analisis kalender, notebook `02_Temporal_Mapping.ipynb` membuat kolom tambahan:

| Kolom | Deskripsi |
|-------|-----------|
| `income_wN_period_start` | Tanggal awal periode minggu ke-N |
| `income_wN_period_end`   | Tanggal akhir periode minggu ke-N (representative date) |
| `income_wN_month`        | Bulan kalender dari period_end |
| `income_wN_year`         | Tahun dari period_end |
| `income_wN_week_of_month`| Minggu ke berapa dalam bulan (dari period_end) |
| `income_wN_iso_week`     | ISO week number dari period_end |

### Definisi week_of_month
| Week | Tanggal |
|------|---------|
| Week 1 | 1–7 |
| Week 2 | 8–14 |
| Week 3 | 15–21 |
| Week 4 | 22–28 |
| Week 5 | 29–31 |

---

## Kolom Utama Dataset

### Survey / Profile
| Kolom | Tipe | Deskripsi | Sumber |
|-------|------|-----------|--------|
| `respondent_id` | string | ID unik responden (R0000–R0383) | Generated |
| `timestamp` | datetime | Waktu pengisian form | Form col 0 |
| `survey_date` | date | Tanggal pengisian form | Derived |
| `usia` | int | Usia responden (clip 17–65) | Form col 2 |
| `gig_type` | string | 8 kategori pekerjaan gig | Form col 4 |
| `domisili_code` | string | Kode domisili (9 region) | Form col 3 |
| `hari_kerja_per_minggu` | int | Hari kerja per minggu (clip 1–7) | Form col 8 |
| `jam_kerja_per_hari` | int | Jam kerja per hari (clip 1–16) | Form col 9 |
| `lama_kerja_bulan` | float | Pengalaman dalam bulan | Form col 7 |

### Income (Survey)
| Kolom | Tipe | Deskripsi |
|-------|------|-----------|
| `income_w1` | float | Pendapatan **minggu lalu** (TERBARU) |
| `income_w2` | float | Pendapatan **2 minggu lalu** |
| `income_w3` | float | Pendapatan **3 minggu lalu** |
| `income_w4` | float | Pendapatan **4 minggu lalu** (TERLAMA) |

### Lag Features (Model)
| Kolom | Tipe | Deskripsi | Leakage Risk |
|-------|------|-----------|--------------|
| `lag_1_income` | float | Income lag 1 (dari urutan kronologis) | Low |
| `lag_2_income` | float | Income lag 2 | Low |
| `lag_3_income` | float | Income lag 3 | Low |
| `lag_4_income` | float | Income lag 4 (terlama) | Low |
| `rolling_mean_4w` | float | Rata-rata 4 lag | Low |
| `rolling_std_4w` | float | Std 4 lag | Low |
| `income_growth_1w` | float | Growth rate lag_1 vs lag_2 | Low |
| `income_volatility` | float | CV dari 4 lag | Low |

### Targets (JANGAN masuk X)
| Kolom | Tipe | Deskripsi | Leakage Risk |
|-------|------|-----------|--------------|
| `next_week_income` | float | **TARGET**: pendapatan minggu depan | **HIGH** |
| `next_week_direction` | string | **TARGET**: Up/Stable/Down | **HIGH** |

### Synthetic
| Kolom | Tipe | Deskripsi |
|-------|------|-----------|
| `synthetic_user_id` | string | ID unik (SYN_000001–SYN_002999) |
| `synthetic_weekly_income` | float | Observed income (sumber lag) |
| `dataset_type` | string | synthetic_52w |

---

## 8 Kategori Gig Type
| Kode | Label Form |
|------|-----------|
| `ojek_online` | Ojek online / driver aplikasi |
| `kurir` | Kurir / pengantar barang atau makanan |
| `jualan_online` | Jualan online / reseller / toko online |
| `freelance_desain` | Freelance desain / editing / ilustrasi |
| `freelance_it` | Freelance IT / website / programming / data |
| `content_creator` | Content creator / admin media sosial |
| `tutor` | Tutor / guru les / pengajar lepas |
| `pekerja_harian` | Pekerja harian / event / part-time |

---

## Direction Classification
- **Up**: perubahan income >= 10% (bukan >)
- **Down**: perubahan income <= -10% (bukan <)
- **Stable**: perubahan antara -10% dan +10%
"""

with open('data_dictionary.md', 'w', encoding='utf-8') as f:
    f.write(data_dict_md)

print('Disimpan: data_dictionary.md')


Disimpan: data_dictionary.md


In [5]:
# CELL 08.5 — Generate notebook.md (alur modular)
notebook_md = """# Notebook Pipeline — Fingo Income Predictor
**Tim:** CC26-PSU217 | **Versi:** v13-FINAL

---

## Alur Modular Notebook

Setiap notebook menarik output dari GitHub di awal (git pull) dan
mendorong hasilnya ke GitHub di akhir (git push).

```
01_Data_Preparation
       ↓ survey_clean.csv (dengan timestamp)
02_Temporal_Mapping
       ↓ survey_temporal_mapped.csv
       ↓ survey_weekly_income_long.csv
03_EDA_Survey
       ↓ charts + survey_eda_summary.md
04_Synthetic_Data_Generation
       ↓ synthetic_52week_user_income.csv
       ↓ synthetic_params.json
05_Feature_Engineering
       ↓ income_features.csv
       ↓ feature_columns.json
06_Model_Dataset_Split
       ↓ income_train.csv / income_val.csv / income_test.csv
       ↓ income_scalers.pkl
       ↓ model_contract.json
07_Bias_Validation
       ↓ bias_validation_report.md + charts
08_Documentation_Export
       ↓ README.md / notebook.md / data_dictionary.md
```

---

## Detail per Notebook

### 01_Data_Preparation.ipynb
**Input:** `data/raw/form_responses.csv`, BPS files
**Proses:**
- Clone/pull repo GitHub
- Setup environment + install library
- Load raw data survey (384 responden)
- Form column mapping berdasarkan posisi kolom
- Drop PII (consent, kontak_gopay)
- **Parse timestamp (DIPERTAHANKAN untuk notebook 02)**
- Convert numerik + clip nilai tidak realistis
- Standardisasi kategori (GIG_MAP, DOMISILI_MAP)
- Multi-hot encoding kolom multi-select
- Feature engineering dasar (avg_weekly_income, income_cv_4w, dll)
- Validasi missing value

**Output:** `data/processed/survey_clean.csv` (dengan timestamp)

---

### 02_Temporal_Mapping.ipynb
**Input:** `data/processed/survey_clean.csv`
**Proses:**
- Mapping `income_w1–income_w4` ke periode kalender berdasarkan timestamp
- `income_w1`: H-7 s/d H-1 dari timestamp responden (terbaru)
- `income_w2`: H-14 s/d H-8
- `income_w3`: H-21 s/d H-15
- `income_w4`: H-28 s/d H-22 (terlama)
- Tambah kolom: period_start, period_end, calendar_month, week_of_month, iso_week
- Buat long-format dataset untuk analisis mingguan

**Output:**
- `data/processed/survey_temporal_mapped.csv`
- `data/processed/survey_weekly_income_long.csv`

---

### 03_EDA_Survey.ipynb
**Input:** survey_temporal_mapped.csv + survey_weekly_income_long.csv
**Proses:**
- Analisis distribusi income per gig_type, domisili, relative_week
- Analisis calendar_month dan week_of_month dari temporal mapping
- Jawab: income tertinggi di relative_week mana? week_of_month berapa?

**Output:** Charts + `outputs/reports/survey_eda_summary.md`

---

### 04_Synthetic_Data_Generation.ipynb
**Input:** `data/processed/survey_temporal_mapped.csv`
**Proses:**
- Generate 3.000 synthetic users dari distribusi survey
- AR(1) income generation dengan noise/shock per gig_type
- Seasonal multiplier (Ramadan, Harbolnas, payday, weekend)

**Output:**
- `data/synthetic/synthetic_52week_user_income.csv`
- `data/synthetic/synthetic_params.json`

---

### 05_Feature_Engineering.ipynb
**Input:** `data/synthetic/synthetic_52week_user_income.csv`
**Proses:**
- Sliding window 4 lag dari 52-week history
- Rolling mean, std, min, max (4w, 2w, 8w)
- Lag features: lag_1 (terbaru) → lag_4 (terlama)
- income_growth_1w, income_volatility, trend_slope_4w
- Calendar features, seasonal flags, OHE gig_type
- Anti-leakage check

**Output:** `data/processed/income_features.csv`

---

### 06_Model_Dataset_Split.ipynb
**Input:** `data/processed/income_features.csv`
**Proses:**
- Kronologis split by synthetic_user_id: 70/15/15
- Fit scaler (log1p → MinMaxScaler) on train only
- Simpan model contract

**Output:** train/val/test CSVs + scalers + model_contract.json

---

### 07_Bias_Validation.ipynb
**Input:** income_features.csv + survey_temporal_mapped.csv + synthetic_52week_user_income.csv
**Proses:**
- Mean vs BPS benchmark
- Distribution test (KS)
- Seasonal direction check
- Autocorrelation lag-1
- BPS range per domisili
- Income per gig_type: synthetic vs survey

**Output:** `outputs/reports/bias_validation_report.md` + charts

---

### 08_Documentation_Export.ipynb
**Input:** semua output notebook sebelumnya
**Output:** `README.md`, `notebook.md`, `data_dictionary.md`

---

## File Final untuk AI Engineer

File di `outputs/model_contract/`:
- `income_train.csv` — dataset training (70%)
- `income_val.csv` — dataset validasi (15%)
- `income_test.csv` — dataset test (15%)
- `income_scalers.pkl` — scaler (target + feature)
- `feature_columns.json` — daftar fitur + metadata
- `model_contract.json` — kontrak lengkap pipeline
"""

with open('notebook.md', 'w', encoding='utf-8') as f:
    f.write(notebook_md)

print('Disimpan: notebook.md')


Disimpan: notebook.md


In [6]:
# CELL 08.6 — Generate README.md
readme_md = """# Fingo — Weekly Income Forecasting for Gig Workers
**Tim:** CC26-PSU217 | **Role:** Data Scientist 2 — Clarisya Adeline
**Branch:** feat/income-predictor-final

---

## Overview
Pipeline prediksi pendapatan mingguan untuk pekerja gig Indonesia.

- **Dataset survey:** 384 responden pekerja gig Indonesia (Google Form Mei 2026)
- **Survey digunakan sebagai:** distribusi acuan untuk generate 3.000 synthetic users
- **Target utama:** weekly income forecasting (prediksi pendapatan minggu depan)
- **Synthetic dataset:** 3.000 users × 52 minggu = 156.000 rows

## Penting: Temporal Mapping income_w1–w4
income_w1–w4 dalam survey **bukan** minggu 1–4 bulan kalender. Mereka adalah
4 periode mingguan relatif sebelum responden mengisi form:
- `income_w1` = H-7 s/d H-1 dari timestamp (terbaru)
- `income_w4` = H-28 s/d H-22 dari timestamp (terlama)

**Urutan kronologis model:** `income_w4 → income_w3 → income_w2 → income_w1`

## Cara Menjalankan
Jalankan notebook secara berurutan dari 01 sampai 08.
Setiap notebook auto-pull dari GitHub di awal dan push ke GitHub di akhir.

```
notebooks/
├── 01_Data_Preparation.ipynb
├── 02_Temporal_Mapping.ipynb
├── 03_EDA_Survey.ipynb
├── 04_Synthetic_Data_Generation.ipynb
├── 05_Feature_Engineering.ipynb
├── 06_Model_Dataset_Split.ipynb
├── 07_Bias_Validation.ipynb
└── 08_Documentation_Export.ipynb
```

## Output untuk AI Engineer
```
outputs/model_contract/
├── income_train.csv
├── income_val.csv
├── income_test.csv
├── income_scalers.pkl
├── feature_columns.json
└── model_contract.json
```

## Dokumentasi
- [notebook.md](notebook.md) — alur modular lengkap
- [data_dictionary.md](data_dictionary.md) — definisi semua kolom
"""

with open('README.md', 'w', encoding='utf-8') as f:
    f.write(readme_md)

print('Disimpan: README.md')
print('\\nSemua dokumentasi berhasil dibuat:')
print('  README.md')
print('  notebook.md')
print('  data_dictionary.md')


Disimpan: README.md
\nSemua dokumentasi berhasil dibuat:
  README.md
  notebook.md
  data_dictionary.md


In [7]:
# CELL 08.7 — Data dictionary CSV (untuk dashboard)
dict_rows = [
    ('respondent_id','string','ID unik responden','Generated','R0000','No','No','Low','R0000–R0383'),
    ('timestamp','datetime','Waktu pengisian form — DIPERTAHANKAN untuk temporal mapping','Form col 0','5/16/2026 10:30:00','No','No','Low','Drop setelah 02_Temporal_Mapping'),
    ('survey_date','date','Tanggal pengisian form','Derived','2026-05-16','Yes','No','Low',''),
    ('gig_type','string','8 kategori pekerjaan gig','Form col 4','ojek_online','Yes','Yes','Low','8 kategori dari GIG_MAP'),
    ('domisili_code','string','Kode domisili 9 region','Form col 3','jabodetabek','Yes','Yes','Low',''),
    ('income_w1','float','Pendapatan MINGGU LALU (terbaru)','Form col 10','500000','Yes','Lag source only','High','Urutan: w4(terlama)→w1(terbaru)'),
    ('income_w2','float','Pendapatan DUA MINGGU LALU','Form col 11','450000','Yes','Lag source only','High',''),
    ('income_w3','float','Pendapatan TIGA MINGGU LALU','Form col 12','480000','Yes','Lag source only','High',''),
    ('income_w4','float','Pendapatan EMPAT MINGGU LALU (terlama)','Form col 13','420000','Yes','Lag source only','High','JANGAN langsung masuk X'),
    ('income_w1_period_start','date','Tgl awal periode income_w1 (H-7)','Temporal mapping','2026-05-09','Yes','No','Low',''),
    ('income_w1_period_end','date','Tgl akhir periode income_w1 (H-1)','Temporal mapping','2026-05-15','Yes','No','Low',''),
    ('income_w1_month','int','Bulan kalender dari income_w1_period_end','Temporal mapping','5','Yes','No','Low',''),
    ('income_w1_week_of_month','int','Week of month (1-5)','Temporal mapping','3','Yes','No','Low','Week1=tgl1-7, Week5=tgl29-31'),
    ('income_w1_iso_week','int','ISO week number','Temporal mapping','20','Yes','No','Low',''),
    ('next_week_income','float','TARGET: pendapatan minggu depan','Engineered','520000','No','Target only','HIGH','JANGAN masuk X'),
    ('next_week_direction','string','TARGET: Up/Stable/Down (threshold >= 10%)','Engineered','Up','No','Target only','HIGH','JANGAN masuk X'),
    ('lag_1_income','float','Income lag 1 minggu (terbaru)','Engineered','500000','No','Yes','Low','Fitur utama'),
    ('lag_4_income','float','Income lag 4 minggu (terlama)','Engineered','420000','No','Yes','Low',''),
    ('rolling_mean_4w','float','Rata-rata 4 lag minggu','Engineered','462500','No','Yes','Low',''),
    ('income_growth_1w','float','Growth rate lag_1 vs lag_2','Engineered','0.11','No','Yes','Low',''),
    ('income_volatility','float','CV dari 4 lag minggu','Engineered','0.07','No','Yes','Low',''),
    ('synthetic_user_id','string','ID unik synthetic user','Generated','SYN_000001','No','No','Low','3.000 unique users'),
    ('synthetic_weekly_income','float','Observed income synthetic (sumber lag)','Synthetic','500000','No','Lag source','High','JANGAN masuk X'),
    ('pref_payday','int','Pref: ramai di tanggal gajian','Form col 15','1','Yes','Yes','Low',''),
    ('bps_jasa_weekly','float','Benchmark BPS jasa per minggu per domisili','BPS','175000','Yes','Yes','Low','Bulanan/4'),
]

df_dict = pd.DataFrame(dict_rows, columns=[
    'column_name','data_type','description','source','example_value',
    'used_for_eda','used_for_modeling','leakage_risk','notes'
])

safe_to_csv(df_dict, 'outputs/reports/data_dictionary.csv')
safe_to_csv(df_dict, 'outputs/dashboard/data_dictionary.csv')

print(f'Data dictionary CSV: {len(df_dict)} entri')
print('  Disimpan: outputs/reports/data_dictionary.csv')
print('  Disimpan: outputs/dashboard/data_dictionary.csv')


Data dictionary CSV: 25 entri
  Disimpan: outputs/reports/data_dictionary.csv
  Disimpan: outputs/dashboard/data_dictionary.csv


In [8]:
# CELL 08.8 — Final check dokumentasi
files_to_check = [
    'README.md',
    'notebook.md',
    'data_dictionary.md',
    'outputs/reports/data_dictionary.csv',
    'outputs/dashboard/data_dictionary.csv',
]

print('=== Documentation Export Check ===')
for path in files_to_check:
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f'{path:45s} | exists={exists} | size={size:,} bytes')

assert os.path.exists('README.md')
assert os.path.exists('notebook.md')
assert os.path.exists('data_dictionary.md')

print('\nDokumentasi final siap.')


=== Documentation Export Check ===
README.md                                     | exists=True | size=1,778 bytes
notebook.md                                   | exists=True | size=4,557 bytes
data_dictionary.md                            | exists=True | size=4,782 bytes
outputs/reports/data_dictionary.csv           | exists=True | size=2,499 bytes
outputs/dashboard/data_dictionary.csv         | exists=True | size=2,499 bytes

Dokumentasi final siap.


In [9]:
# GIT PUSH — Commit dan push output notebook ini ke GitHub
import os, subprocess

LOCAL_DIR   = "/content/fingo-income-analysis"
BRANCH_NAME = "feat/income-predictor-final"
NOTEBOOK_NAME = "08_Documentation_Export.ipynb"

os.chdir(LOCAL_DIR)

def run_cmd(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(f"$ {cmd}")
    if r.stdout.strip(): print(r.stdout.strip())
    if r.stderr.strip(): print(r.stderr.strip())
    if check and r.returncode != 0:
        raise RuntimeError(f"Command gagal: {cmd}")
    return r

run_cmd('git config user.email "adelineclarisya@gmail.com"')
run_cmd('git config user.name "ClarisyaA"')

print("\n[1] Cek status")
run_cmd("git status --short", check=False)

print("\n[2] Add semua perubahan output")
run_cmd("git add data/ outputs/ notebooks/ *.ipynb README.md notebook.md data_dictionary.md", check=False)

print("\n[3] Commit")
commit_result = run_cmd(
    f'git commit -m "docs(DS2): generate documentation from {NOTEBOOK_NAME}"',
    check=False
)
if commit_result.returncode != 0:
    print("[INFO] Tidak ada perubahan baru, skip commit.")

print("\n[4] Fetch remote terbaru")
run_cmd("git fetch origin")

print("\n[5] Rebase lalu push")
run_cmd(f"git pull --rebase origin {BRANCH_NAME}")
run_cmd(f"git push -u origin {BRANCH_NAME}")

print("\nPush berhasil!")


$ git config user.email "adelineclarisya@gmail.com"
$ git config user.name "ClarisyaA"

[1] Cek status
$ git status --short
M README.md
 M notebook.md

[2] Add semua perubahan output
$ git add data/ outputs/ notebooks/ *.ipynb README.md notebook.md data_dictionary.md

[3] Commit
$ git commit -m "docs(DS2): generate documentation from 08_Documentation_Export.ipynb"
[feat/income-predictor-final f75b054] docs(DS2): generate documentation from 08_Documentation_Export.ipynb
 2 files changed, 10 insertions(+), 10 deletions(-)

[4] Fetch remote terbaru
$ git fetch origin

[5] Rebase lalu push
$ git pull --rebase origin feat/income-predictor-final
Current branch feat/income-predictor-final is up to date.
From https://github.com/ClarisyaA/fingo-income-analysis
 * branch            feat/income-predictor-final -> FETCH_HEAD
$ git push -u origin feat/income-predictor-final
Branch 'feat/income-predictor-final' set up to track remote branch 'feat/income-predictor-final' from 'origin'.
To https://git